# Chapter 07 — Entries, Text widgets, validation, and form design

**Book role:** Core GUI  
**Suggested book space:** about 9 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Build single-line and multiline inputs.
- Validate input before performing calculations or queries.
- Distinguish keystroke validation from submission-time validation.
- Design keyboard-friendly forms with actionable error messages.

## 1. Entry and Text serve different tasks

`ttk.Entry` is suitable for compact single-line values: names, amounts, IDs, paths, and search terms. `tk.Text` handles multiline notes, SQL previews, logs, and report text. The Text widget uses special indices such as `"1.0"` for line 1, character 0 and `"end-1c"` to avoid including the widget’s implicit trailing newline when reading text.

A form should specify which fields are required, expected formats, allowed ranges, and normalization rules. A blank optional text field, an invalid integer, and a valid zero are three different states. Avoid treating a truthiness check as validation: `0` and `""` can have different meanings in analytics.

### Worked example 1: Read and validate a form

The form collects and validates first, then performs an action only on success. Real applications should call a service or repository after validation and handle its errors separately.

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox

root = tk.Tk()
root.title("Validated input")
root.geometry("430x210")
name_var = tk.StringVar()
amount_var = tk.StringVar()

form = ttk.Frame(root, padding=18)
form.pack(fill="both", expand=True)
for row, (label, var) in enumerate((("Product", name_var), ("Unit price", amount_var))):
    ttk.Label(form, text=label).grid(row=row, column=0, sticky="w", padx=(0, 12), pady=6)
    ttk.Entry(form, textvariable=var).grid(row=row, column=1, sticky="ew", pady=6)
form.columnconfigure(1, weight=1)


def submit() -> None:
    name = name_var.get().strip()
    raw_amount = amount_var.get().strip()
    errors = []
    if not name:
        errors.append("Product is required.")
    try:
        amount = float(raw_amount)
        if amount < 0:
            errors.append("Unit price cannot be negative.")
    except ValueError:
        errors.append("Unit price must be a number, such as 12.50.")
    if errors:
        messagebox.showerror("Check the form", "\n".join(errors), parent=root)
        return
    print({"product": name, "unit_price": amount})
    messagebox.showinfo("Validated", "The values are ready to save.", parent=root)

ttk.Button(form, text="Validate", command=submit).grid(row=2, column=1, sticky="e", pady=10)
root.mainloop()

## 2. Validate at the appropriate boundary

Keystroke-level validation can prevent a user from entering a forbidden character, but it can also make legitimate edits difficult (for example, temporarily deleting a field while replacing a number). Submission-time validation is better for most multi-field business rules: read all values, normalize them, validate the complete record, then either show actionable errors or call the service layer.

Always validate at more than one boundary. GUI validation improves feedback, but imported CSV/Excel data and direct service calls can bypass the form. The data-cleaning layer should independently verify types, nullability, uniqueness, ranges, and domain constraints. A GUI’s validation is usability support, not a security boundary.

### Worked example 2: Get text and remove the implicit trailing newline

Use Text for multiline content, not a large Entry. The `repr` output makes newlines and trailing characters visible during debugging.

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
text = tk.Text(root, width=48, height=6, wrap="word")
text.pack(padx=12, pady=12)
text.insert("1.0", "A multiline note.\nAdd another line.")


def show_value() -> None:
    value = text.get("1.0", "end-1c")
    print(repr(value))

ttk.Button(root, text="Read note", command=show_value).pack(pady=(0, 12))
root.mainloop()

## 3. Error messages that help users recover

A message such as “Invalid value” is not enough. Identify the field, explain the accepted format, preserve the user’s entered value when practical, and move focus to the first invalid field. Avoid showing a modal dialog for every small validation error during typing; an adjacent message or status region is often better. Use dialogs for errors that require attention or may lead to data loss.

Normalize human-entered values at the boundary: trim surrounding whitespace, use a defined decimal separator policy, parse dates explicitly, and keep raw and parsed values separate if auditability matters. Do not silently turn a malformed amount into zero or a failed date into today’s date.

## Guided lab: Lab: a data-entry form

Create a customer or transaction-entry form with required fields, numeric and date validation, field-level recovery, keyboard tab order, and a submission summary. Test empty input, whitespace-only input, malformed numbers, negative values, an extremely large amount, and correct data. Write the domain rules before writing the widgets.

## Review questions and practice

1. Why is input validation needed even if the field is a numeric Tk variable?
2. What is the difference between `end` and `end-1c` for Text.get?
3. Why should imported files use validation outside the GUI?
4. What properties make an error message actionable?

## Further reading

- Tkinter Entry and Text: https://docs.python.org/3/library/tkinter.html
- Python numeric conversion: https://docs.python.org/3/library/functions.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.